In [1]:
import torch
import torch.nn as nn 
import torchvision.transforms.functional as TF




class DoubleConv(nn.Module):
    def __init__(self, in_channel,out_channel):
        super(DoubleConv,self).__init__()
        self.conv = nn.sequential(
        nn.conv2d(in_channel,out_channel,3,1,1,bias= False),
        nn.BatchNorm2d(out_channel),
        nn.ReLU(inplace = True),
        nn.conv2d(in_channel,out_channel,3,1,1,bias= False),
        nn.BatchNorm2d(out_channel),
        nn.ReLU(inplace = True)
        )
    def forward(self,x):
        return self.conv(x)
        


In [2]:
class U_Net(nn.Module):
    def __init__(self,in_channel=3,out_channel=1,features =[64,128,256,512]):
        super(U_Net,self).__init__()
        self.ups = nn.ModuleList()
        self.downs = nn.ModuleList()
        self.pool = nn.MaxPool2d(2,2)

        ### for downs 
        for feature in features:
            self.downs.append(DoubleConv(in_channel,feature))
            in_channel = feature
        ##for ups 
        for feature in reversed(features):
            self.ups.append(ConvTranspose2d(feature*2,feature,kernal_size = 2, stride = 2))
            self.ups.append(DoubleConv(feature*2,feature))
        self.bottleneck = nn.Conv2d(features[-1],features[-1]*2,3,1,1)
        self.final_conv = nn.Conv2d(features[0],out_channel,3,1,1)

    def forward(self,x):
        ### for down 
        skip_connections = []
        for down in self.downs: 
            x = self.down(x)
            skip_connections.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        ### for up 
        skip_connection = skip_connection[::-2]
        for idx in range(0,len(self.ups), 2):
            x = self.ups[idx](x)

            if x.shape!= skip_connection[idx].shape:
                x= TF.reshape(x,size= skip_connection.shape[2:])
            skip_value = torch.cat((skip_connection[idx//2],x),dim=1)
            x= self.ups[idx+1](skip_value)
        return self.final_conv(x)

In [3]:
def Test():
    x = torch.randn(3,1,160,160)
    model = U_Net(in_channel= 1, out_channel=1)
    preds = model(x)
    print(preds)

In [4]:
if __name__=="main":
    Test()